# 05 - Pandas: GROUP BY, JOIN, and Reshaping

**Prerequisite:** notebook 04.
**This notebook:** `groupby` in all its forms, every join type, and pivot / melt / stack.

If you know SQL well, this is the notebook where Pandas starts to feel familiar - and where the differences from SQL matter most.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 25)

DATA = next(p for p in [Path("../data"), Path("data"), Path("study/data")] if p.exists())
orders    = pd.read_csv(DATA / "orders.csv", parse_dates=["order_ts"])
customers = pd.read_csv(DATA / "customers.csv", parse_dates=["signup_date"])
items     = pd.read_csv(DATA / "order_items.csv")
print(orders.shape, customers.shape, items.shape)

## 1. groupby = split, apply, combine

Three steps, always:

1. **split** the rows into groups by one or more keys
2. **apply** a function to each group
3. **combine** the results into a new object

Java 8: `stream().collect(Collectors.groupingBy(Order::getChannel, Collectors.summingDouble(Order::getAmount)))`.
SQL: `SELECT channel, SUM(amount) FROM orders GROUP BY channel`.

In [ ]:
g = orders.groupby("channel")
print(type(g))
print("groups:", list(g.groups.keys()))
print("group sizes:")
print(g.size())                         # COUNT(*) per group

In [ ]:
# One aggregate
print(orders.groupby("channel")["amount"].sum().round(2))
print()
# Several aggregates on one column
print(orders.groupby("channel")["amount"].agg(["count", "sum", "mean", "min", "max"]).round(2))

> Note `count` vs `size`. `size()` counts **rows** (including nulls); `count()` counts **non-null values** - exactly the `COUNT(*)` vs `COUNT(col)` distinction in SQL.

In [ ]:
chk = orders.groupby("channel").agg(rows=("amount", "size"), non_null=("amount", "count"))
print(chk)

### Named aggregation - the readable form

`agg(new_name=("column", "function"))` is the clearest syntax and produces flat column names. Prefer it.

In [ ]:
summary = (orders
    .groupby("channel")
    .agg(
        n_orders    = ("order_id", "count"),
        total_amt   = ("amount", "sum"),
        avg_amt     = ("amount", "mean"),
        max_amt     = ("amount", "max"),
        n_customers = ("customer_id", "nunique"),
        first_order = ("order_ts", "min"),
    )
    .round({"total_amt": 2, "avg_amt": 2, "max_amt": 2})
    .sort_values("total_amt", ascending=False)
)
print(summary)

The equivalent SQL:

```sql
SELECT channel,
       COUNT(order_id)          AS n_orders,
       SUM(amount)              AS total_amt,
       AVG(amount)              AS avg_amt,
       MAX(amount)              AS max_amt,
       COUNT(DISTINCT customer_id) AS n_customers,
       MIN(order_ts)            AS first_order
FROM orders
GROUP BY channel
ORDER BY total_amt DESC;
```

In [ ]:
# Custom aggregate: any function that takes a Series and returns a scalar
def p90(s):
    return s.quantile(0.90)

print(orders.groupby("channel").agg(
    median_amt = ("amount", "median"),
    p90_amt    = ("amount", p90),
    spread     = ("amount", lambda s: s.max() - s.min()),
).round(2))

## 2. Grouping by several keys

In [ ]:
multi = orders.groupby(["channel", "status"])["amount"].sum().round(0)
print(multi.head(8))
print()
print("the result has a MultiIndex:", multi.index.names)

In [ ]:
# as_index=False gives a flat table, which is usually what you want downstream
flat = orders.groupby(["channel", "status"], as_index=False)["amount"].sum().round(0)
print(flat.head())
print()
# unstack turns the inner index level into columns - a pivot
print(multi.unstack(fill_value=0))

## 3. `transform` - aggregate without collapsing rows

This is the single most useful Pandas idea for anyone from SQL: `transform` is the **window function**.

`agg` gives one row per group. `transform` gives one row per **original row**, filled with that row's group value - exactly like `SUM(x) OVER (PARTITION BY k)`.

In [ ]:
o = orders.copy()
o["channel_total"] = o.groupby("channel")["amount"].transform("sum")
o["channel_avg"]   = o.groupby("channel")["amount"].transform("mean")
o["share_of_channel"] = (o["amount"] / o["channel_total"] * 100).round(3)

print(o[["order_id", "channel", "amount", "channel_total", "channel_avg", "share_of_channel"]].head(6).round(2))

SQL equivalent:

```sql
SELECT order_id, channel, amount,
       SUM(amount) OVER (PARTITION BY channel) AS channel_total,
       AVG(amount) OVER (PARTITION BY channel) AS channel_avg
FROM orders;
```

In [ ]:
# Common transform recipes
o["rank_in_channel"] = o.groupby("channel")["amount"].rank(ascending=False, method="dense")   # DENSE_RANK()
o["row_num"]         = o.groupby("channel").cumcount() + 1                                   # ROW_NUMBER()
o["z_in_channel"]    = o.groupby("channel")["amount"].transform(lambda s: (s - s.mean()) / s.std())

print(o.sort_values(["channel", "rank_in_channel"])
       [["channel", "order_id", "amount", "rank_in_channel", "row_num", "z_in_channel"]]
       .head(6).round(3))

In [ ]:
# Fill missing values with the group's own mean - a classic transform use
o["amount_filled"] = o["amount"].fillna(o.groupby("channel")["amount"].transform("mean"))
print("nulls before:", o["amount"].isna().sum(), " after:", o["amount_filled"].isna().sum())

## 4. `filter` - your `HAVING` clause

`groupby(...).filter(fn)` keeps **all rows** of the groups where `fn(group)` is True.

In [ ]:
busy = orders.groupby("customer_id").filter(lambda g: len(g) >= 18)
print("customers with >= 18 orders:", busy["customer_id"].nunique())
print("rows kept:", len(busy), "of", len(orders))
print()
# The agg-style equivalent (HAVING on the summary itself)
counts = orders.groupby("customer_id").size()
print("same customers via HAVING on the aggregate:", (counts >= 18).sum())

## 5. `apply` - the general escape hatch

`apply` passes the whole sub-DataFrame to your function. It is flexible and **slow** - it runs a Python call per group. Use it only when `agg` and `transform` cannot express the operation.

In [ ]:
def top_two_orders(g):
    return g.nlargest(2, "amount")[["order_id", "amount"]]

top2 = orders.groupby("channel")[["order_id", "amount"]].apply(top_two_orders, include_groups=False)
print(top2)

> **Performance order, best first:** built-in string aggregates (`"sum"`, `"mean"`) -> `transform` with a built-in -> `agg` with a small lambda -> `apply`. The built-ins run in C over the whole column; `apply` runs Python once per group.

## 6. Joins with `merge`

`df.merge(other, ...)` is `JOIN`. The important arguments:

| Argument | Meaning |
|---|---|
| `how` | `inner` (default), `left`, `right`, `outer`, `cross` |
| `on` | key column present in both |
| `left_on` / `right_on` | keys with different names |
| `left_index=True` | join using the index instead of a column |
| `suffixes=("_x","_y")` | how to rename colliding column names |
| `indicator=True` | adds a `_merge` column: `left_only` / `right_only` / `both` |
| `validate="m:1"` | **assert** the cardinality; raises if violated |

Pandas `merge` defaults to `inner`. SQL's bare `JOIN` also means inner, so that matches.

In [ ]:
enriched = orders.merge(customers, on="customer_id", how="left")
print("orders:", len(orders), "-> after LEFT JOIN customers:", len(enriched))
print(enriched[["order_id", "customer_id", "name", "city", "tier", "amount"]].head())

In [ ]:
# Compare the join types on a tiny example
left  = pd.DataFrame({"k": ["a", "b", "c"], "lv": [1, 2, 3]})
right = pd.DataFrame({"k": ["b", "c", "d"], "rv": [20, 30, 40]})

for how in ["inner", "left", "right", "outer"]:
    print(f"--- how='{how}' ---")
    print(left.merge(right, on="k", how=how).to_string(index=False))
    print()

In [ ]:
# cross join = CROSS JOIN (every combination). Watch the row count.
sizes  = pd.DataFrame({"size": ["S", "M", "L"]})
colors = pd.DataFrame({"color": ["red", "blue"]})
print(sizes.merge(colors, how="cross").to_string(index=False))

### `indicator` and `validate` - use these

`indicator=True` answers "which rows did not match?" - the Pandas version of an anti-join check.
`validate=` turns a silent row explosion into a loud error. **Always pass it on important joins.**

In [ ]:
chk = left.merge(right, on="k", how="outer", indicator=True)
print(chk.to_string(index=False))
print()
print("anti-join (rows only in left):")
print(chk[chk["_merge"] == "left_only"][["k", "lv"]].to_string(index=False))

In [ ]:
# validate catches the classic bug: a key you thought was unique, is not
dupes = pd.DataFrame({"k": ["a", "a", "b"], "rv": [1, 2, 3]})
print("without validate, rows silently multiply:", len(left.merge(dupes, on="k")))
try:
    left.merge(dupes, on="k", validate="1:1")
except Exception as e:
    print("with validate='1:1':", type(e).__name__, "-", str(e)[:80])

> **The most common Pandas data bug:** a many-to-many join that multiplies rows. `orders` joined to `order_items` (1 order -> many items) legitimately expands rows; a join on a key you *assumed* was unique does so silently and wrongly. `validate="1:1"`, `"1:m"`, `"m:1"` costs nothing and catches it.

In [ ]:
# A real multi-table query: revenue per city, delivered orders only
report = (orders
    .query("status == 'DELIVERED'")
    .merge(customers, on="customer_id", how="inner", validate="m:1")
    .groupby("city", dropna=False)
    .agg(n_orders=("order_id", "count"),
         revenue=("amount", "sum"),
         avg_order=("amount", "mean"))
    .round(2)
    .sort_values("revenue", ascending=False)
)
print(report)

### `concat` - `UNION ALL` and column-wise stacking

In [ ]:
jan = orders[orders["order_ts"].dt.month == 1].head(3)
feb = orders[orders["order_ts"].dt.month == 2].head(3)

stacked = pd.concat([jan, feb], ignore_index=True)        # UNION ALL
print("rows:", len(stacked))
print(stacked[["order_id", "order_ts", "amount"]].to_string(index=False))
print()
print("side by side (axis=1) aligns on the INDEX, not on position:")
print(pd.concat([jan[["order_id"]].reset_index(drop=True),
                 feb[["amount"]].reset_index(drop=True)], axis=1).to_string(index=False))

> `pd.concat([...], axis=1)` aligns on the index. If the frames have unrelated indexes you get a sparse mess full of NaN. Reset the index first, or use `merge` with a real key.

## 7. Reshaping: wide vs long

- **long / tidy**: one row per observation, e.g. `(date, metric, value)`. Good for storage and for groupby.
- **wide / pivoted**: one row per entity, one column per category. Good for reading and for reports.

`pivot_table` goes long -> wide. `melt` goes wide -> long.

In [ ]:
pt = pd.pivot_table(
    orders,
    index="channel",          # rows
    columns="status",         # columns
    values="amount",          # what to aggregate
    aggfunc="sum",            # how
    fill_value=0,
    margins=True,             # adds an "All" row and column, like GROUP BY ROLLUP
).round(0)
print(pt)

In [ ]:
# Several aggregates at once
pt2 = pd.pivot_table(orders, index="channel", columns="status",
                     values="amount", aggfunc=["count", "mean"], fill_value=0).round(1)
print(pt2)

> `pivot` vs `pivot_table`: `pivot` only reshapes and **raises** on duplicate index/column pairs; `pivot_table` aggregates them. When in doubt use `pivot_table`.

In [ ]:
# melt: wide -> long
wide = pd.pivot_table(orders, index="channel", columns="status", values="amount",
                      aggfunc="sum", fill_value=0).round(0).reset_index()
print("wide:")
print(wide.to_string(index=False))
print()
long = wide.melt(id_vars="channel", var_name="status", value_name="total_amount")
print("long:")
print(long.head(6).to_string(index=False))

In [ ]:
# crosstab: a frequency table in one call
print(pd.crosstab(orders["channel"], orders["status"]))
print()
print("as row percentages:")
print((pd.crosstab(orders["channel"], orders["status"], normalize="index") * 100).round(1))

## 8. MultiIndex - hierarchical labels

A `groupby` on several keys produces a MultiIndex. `stack` and `unstack` move levels between the index and the columns.

In [ ]:
mi = orders.groupby(["channel", "status"])["amount"].sum().round(0)
print(mi.head(6))
print()
print("select one outer key:")
print(mi.loc["web"])
print()
print("select an exact pair:", mi.loc[("web", "DELIVERED")])
print()
print("cross-section on an inner level (all channels, DELIVERED only):")
print(mi.xs("DELIVERED", level="status"))

In [ ]:
unst = mi.unstack()          # inner index level -> columns
print("unstacked:\n", unst)
print()
print("stack() puts it back:\n", unst.stack().head(4))
print()
print("flatten a MultiIndex column header:")
wide2 = orders.groupby("channel").agg({"amount": ["sum", "mean"]}).round(1)
wide2.columns = ["_".join(c) for c in wide2.columns]
print(wide2)

## 9. Exercises

**E1.** Revenue and order count per customer `tier` (join needed), delivered orders only, sorted by revenue descending.

**E2.** For each customer, add a column with their rank by total spend across all customers. (Hint: aggregate, then `rank`.)

**E3.** For each order, add the customer's average order amount **without** collapsing rows.

**E4.** Which SKUs appear in the most distinct orders? Use `order_items`.

**E5.** Build a wide table: rows = month, columns = channel, values = total revenue.

**E6.** Find customers in `customers.csv` who have never placed an order (anti-join).

In [ ]:
# Solutions
e1 = (orders.query("status == 'DELIVERED'")
      .merge(customers, on="customer_id", validate="m:1")
      .groupby("tier", dropna=False)
      .agg(revenue=("amount", "sum"), n=("order_id", "count"))
      .round(2).sort_values("revenue", ascending=False))
print("E1:\n", e1, "\n")

spend = orders.groupby("customer_id")["amount"].sum().rename("total_spend").reset_index()
spend["spend_rank"] = spend["total_spend"].rank(ascending=False, method="dense").astype(int)
print("E2:\n", spend.sort_values("spend_rank").head(5).round(2).to_string(index=False), "\n")

e3 = orders.assign(cust_avg=lambda d: d.groupby("customer_id")["amount"].transform("mean").round(2))
print("E3:\n", e3[["order_id", "customer_id", "amount", "cust_avg"]].head(4).to_string(index=False), "\n")

e4 = items.groupby("sku")["order_id"].nunique().sort_values(ascending=False)
print("E4:\n", e4, "\n")

e5 = (orders.assign(month=lambda d: d["order_ts"].dt.to_period("M").astype(str))
      .pivot_table(index="month", columns="channel", values="amount", aggfunc="sum", fill_value=0)
      .round(0))
print("E5 (first 4 months):\n", e5.head(4), "\n")

e6 = customers.merge(orders[["customer_id"]].drop_duplicates(), on="customer_id",
                     how="left", indicator=True)
print("E6 customers with no orders:",
      e6[e6["_merge"] == "left_only"]["name"].tolist() or "none - every customer ordered")

## Recap

| SQL | Pandas |
|---|---|
| `GROUP BY k` + aggregate | `df.groupby("k").agg(...)` |
| `COUNT(*)` / `COUNT(col)` | `size()` / `count()` |
| `COUNT(DISTINCT col)` | `nunique()` |
| `HAVING` | `groupby(...).filter(lambda g: ...)` |
| `SUM(x) OVER (PARTITION BY k)` | `groupby("k")["x"].transform("sum")` |
| `ROW_NUMBER() OVER (PARTITION BY k)` | `groupby("k").cumcount() + 1` |
| `RANK` / `DENSE_RANK` | `groupby("k")["x"].rank(method="min" / "dense")` |
| `INNER/LEFT/RIGHT/FULL JOIN` | `merge(..., how=...)` |
| `UNION ALL` | `pd.concat([...])` |
| `PIVOT` | `pivot_table` |
| `UNPIVOT` | `melt` |
| `GROUP BY ROLLUP` | `pivot_table(..., margins=True)` |

Two habits worth forming: use **named aggregation** for readable output, and pass **`validate=`** on every merge.

**Next:** `06_pandas_timeseries_and_windows.ipynb`.